# MRS-SHAP: Multi-Objective Explainable EEG Feature Selection Framework for ADHD Detection

This notebook implements the complete preprocessing, feature extraction, feature selection, and classification pipeline described in the paper:

> **MRS-SHAP: Multi-objective explainable EEG feature selection framework for ADHD detection**  
> Mohammed Atheef G A, Shyam Pranav G, and Omkar S Powar  
> *Discover Applied Sciences* (2026)  
> DOI: [10.1007/s42452-026-08788-7](https://doi.org/10.1007/s42452-026-08788-7)

The framework integrates SHAP-based feature importance, Dynamic Time Warping (DTW)-based temporal stability, and performance-aware weighting (BW) into a unified ranking score (MRS-SHAP) for identifying robust and neurophysiologically relevant EEG features for general ADHD classification.


## 1. Install Dependencies

Run the cell below to install the necessary libraries.


In [ ]:
!pip install catboost xgboost shap mne pywavelets dtaidistance imbalanced-learn statsmodels matplotlib seaborn psutil tqdm --quiet


## 2. Configuration & Dataset Paths

Specify the paths to your local folders or Google Drive folders containing the dataset MAT files.


In [ ]:
import os

# ==========================================
# DATASET PATHS
# ==========================================
# Default directories on local system. Please modify these if your dataset is located elsewhere.
ADHD_PART1 = os.environ.get("ADHD_PART1", "dataset/ADHD_part1")
ADHD_PART2 = os.environ.get("ADHD_PART2", "dataset/ADHD_part2")
CONTROL_PART1 = os.environ.get("CONTROL_PART1", "dataset/Control_part1")
CONTROL_PART2 = os.environ.get("CONTROL_PART2", "dataset/Control_part2")

# ==========================================
# SIGNAL ACQUISITION PARAMETERS
# ==========================================
FS = 128                 # Sampling frequency (Hz)
WINDOW_SIZE = 512        # Sliding window size (epochs)
STRIDE = 256            # Stride size (50% overlap)
EXPECTED_CHANNELS = 19   # Standard 10-20 EEG configuration

# Channel names matching the 10-20 standard placement system
CHANNEL_NAMES = [
    "Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", 
    "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", 
    "Fz", "Cz", "Pz"
]



## 3. Preprocessing Steps

Implements IIR notch filtering (50/60 Hz), spike clipping, Butterworth bandpass filtering (1-45 Hz), and multi-metric weighted ICA artifact removal.


In [ ]:
import time
import numpy as np
from scipy import signal
from scipy.stats import kurtosis
import mne
from tqdm import tqdm

def notch_filter(data, fs=128, freqs=[50], Q=30):
    """Apply IIR notch filter to remove power-line interference."""
    data_filtered = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq, Q, fs)
        # Apply along the time axis (axis 1 for shape: segments x samples x channels)
        data_filtered = signal.filtfilt(b, a, data_filtered, axis=1)
    return data_filtered

def clip_spikes(data, threshold=500):
    """Clip transient spikes exceeding threshold."""
    return np.clip(data, -threshold, threshold)

def butterworth_filter(data, fs=128, lowcut=1.0, highcut=45.0, order=4):
    """Apply Butterworth bandpass filter to EEG data."""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, data, axis=1)

def apply_ica(data, fs=128, n_components=None, method='weighted', weights=None):
    """Apply ICA with weighted (kurtosis, variance, peak, Fp2, frequency) or standard method."""
    start_time = time.time()
    print(f"Applying ICA (method={method}) with input shape={data.shape}")
    
    if data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim != 3:
        raise ValueError(f"Expected 3D data, got {data.ndim}D")
        
    n_segments, n_samples, n_channels = data.shape
    if n_components is None or n_components > n_channels:
        n_components = min(19, n_channels)
        
    print(f"Using n_components={n_components}")
    if n_components < 1:
        print("Warning: n_components < 1, skipping ICA exclusion")
        return data, data
        
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['EEG1', 'EEG2', 'EEG3', 'EEG4', 'EEG5', 'EEG6', 'EEG7', 'EEG8', 'EEG9', 'EEG10', 'EEG11', 'EEG12', 'EEG13', 'EEG14', 'EEG15', 'EEG16', 'EEG17', 'EEG18', 'EEG19']
    
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    
    try:
        theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162]
        radius_values_cm = [51.3, 51.3, 51.3, 33.3, 25.6, 33.3, 51.3, 51.3, 25.6, 0, 25.6, 51.3, 51.3, 33.3, 25.6, 33.3, 51.3, 51.3, 51.3]
        radius_values_m = [r / 100 for r in radius_values_cm]
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0]
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception:
        print("Warning: Failed to set custom montage, using standard_1020")
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)
        
    # Highpass at 0.5Hz for ICA stability
    raw.filter(l_freq=0.5, h_freq=45.0, method='iir', iir_params=dict(order=4, ftype='butter'), verbose=False)
    
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw, verbose=False)
    
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.mean(np.abs(ica_sources[:, 1])) / np.mean(np.abs(ica_sources)) if n_channels > 1 else 0
    
    nperseg = min(total_samples, 512)
    f, psd = signal.welch(ica_sources.T, fs=fs, nperseg=nperseg, axis=1)
    artifact_band = (f >= 45) & (f <= 60)
    artifact_power = np.mean(psd[:, artifact_band], axis=1) + 1e-10
    
    if method == 'weighted':
        if weights is None:
            weights = {'kurtosis': 0.45, 'variance': 0.15, 'peak': 0.15, 'fp2': 0.10, 'artifact': 0.15}
            
        kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
        variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
        peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
        artifact_norm = (artifact_power - artifact_power.min()) / (artifact_power.max() - artifact_power.min() + 1e-10)
        
        combined_score = (weights['kurtosis'] * kurtosis_norm +
                         weights['variance'] * variance_norm +
                         weights['peak'] * peak_norm +
                         weights['fp2'] * fp2_weight +
                         weights['artifact'] * artifact_norm)
                         
        score_threshold = np.percentile(combined_score, 90)
        top_artifact_indices = np.where(combined_score > score_threshold)[0].tolist()
        if not top_artifact_indices or len(top_artifact_indices) < 2:
            top_artifact_indices = np.argsort(combined_score)[-min(2, n_components):].tolist()
        ica.exclude = top_artifact_indices
    else:
        score_threshold = np.percentile(kurtosis_values, 90)
        top_artifact_indices = np.where(kurtosis_values > score_threshold)[0].tolist()
        if not top_artifact_indices:
            top_artifact_indices = np.argsort(kurtosis_values)[-min(2, n_components):].tolist()
        ica.exclude = top_artifact_indices
        
    raw_clean = ica.apply(raw.copy(), verbose=False)
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    
    print(f"ICA ({method}): Time={time.time()-start_time:.2f}s, Output shape={cleaned_data.shape}")
    return cleaned_data, data

def compare_ica_methods(X_raw, X_cleaned_ica, X_cleaned_weighted, fs=128):
    """Compare standard ICA and weighted ICA using SNR in multiple EEG bands."""
    nperseg = min(X_raw.shape[1], 512)
    
    f, psd_raw = signal.welch(X_raw, fs=fs, nperseg=nperseg, axis=1, scaling='density')
    _, psd_ica = signal.welch(X_cleaned_ica, fs=fs, nperseg=nperseg, axis=1, scaling='density')
    _, psd_weighted = signal.welch(X_cleaned_weighted, fs=fs, nperseg=nperseg, axis=1, scaling='density')
    
    signal_bands = [(1, 4), (4, 8), (8, 13), (13, 30)]
    noise_band = (45, 60)
    
    signal_power_ica = 0
    signal_power_weighted = 0
    band_weights = [0.2, 0.3, 0.3, 0.2]
    
    for (low, high), w in zip(signal_bands, band_weights):
        band_mask = (f >= low) & (f <= high)
        signal_power_ica += np.mean(psd_ica[:, band_mask, :], axis=1) * w
        signal_power_weighted += np.mean(psd_weighted[:, band_mask, :], axis=1) * w
        
    signal_power_ica = signal_power_ica + 1e-10
    signal_power_weighted = signal_power_weighted + 1e-10
    
    noise_mask = (f >= noise_band[0]) & (f <= noise_band[1])
    noise_power_ica = np.mean(psd_ica[:, noise_mask, :], axis=1) + 1e-9
    noise_power_weighted = np.mean(psd_weighted[:, noise_mask, :], axis=1) + 1e-9
    
    snr_ica = 10 * np.log10(np.clip(signal_power_ica / noise_power_ica, 1e-10, 1e10))
    snr_weighted = 10 * np.log10(np.clip(signal_power_weighted / noise_power_weighted, 1e-10, 1e10))
    
    snr_ica_mean = np.mean(snr_ica)
    snr_weighted_mean = np.mean(snr_weighted)
    
    print(f"Standard ICA SNR (mean): {snr_ica_mean:.4f} dB")
    print(f"Weighted ICA SNR (mean): {snr_weighted_mean:.4f} dB")
    return snr_ica_mean, snr_weighted_mean

def find_optimal_weights(X, X_raw, fs=128, n_components=19):
    """Perform random search to find optimal weights for Weighted ICA, maximizing SNR."""
    print("Starting random search for optimal weights...")
    start_time = time.time()
    
    n_iterations = 15  # Scaled down slightly for faster execution, can be configured
    best_snr_weighted = -np.inf
    best_snr_diff = -np.inf
    best_weights = None
    
    for i in range(n_iterations):
        weights = {
            'kurtosis': np.random.uniform(0.4, 0.5),
            'variance': np.random.uniform(0.1, 0.2),
            'peak': np.random.uniform(0.15, 0.25),
            'fp2': np.random.uniform(0.05, 0.15),
            'artifact': np.random.uniform(0.15, 0.25)
        }
        total = sum(weights.values())
        weights = {k: v / total for k, v in weights.items()}
        try:
            X_cleaned_weighted, _ = apply_ica(X, fs=fs, n_components=n_components, method='weighted', weights=weights)
            X_cleaned_ica, _ = apply_ica(X, fs=fs, n_components=n_components, method='standard')
            snr_ica, snr_weighted = compare_ica_methods(X_raw, X_cleaned_ica, X_cleaned_weighted, fs=fs)
            snr_diff = snr_weighted - snr_ica
            if snr_weighted > best_snr_weighted:
                best_snr_weighted = snr_weighted
                best_snr_diff = snr_diff
                best_weights = weights
                print(f"Iter {i+1} best composite SNR: {snr_weighted:.4f} dB (Diff over standard: {snr_diff:.4f} dB)")
        except Exception as e:
            print(f"Iteration {i+1} failed: {str(e)}")
            continue
            
    print(f"Random search completed in {time.time()-start_time:.2f}s")
    return best_weights, best_snr_weighted, best_snr_diff



## 4. Feature Extraction

Extracts all 998 features across Spectral, Nonlinear, and Global domains.


In [ ]:
import time
import numpy as np
from scipy import signal
from scipy.signal import hilbert, find_peaks
from sklearn.cluster import KMeans
import pywt
from tqdm import tqdm

# Import preprocessing filters
from preprocessing import butterworth_filter

def calculate_psd(data, fs=128, freq_range=None):
    """Calculate power spectral density using Welch's method."""
    nperseg = min(len(data), 512)
    f, psd = signal.welch(data, fs=fs, nperseg=nperseg, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0) if freq_mask.any() else 0
    return psd.mean(axis=0)

def calculate_engagement_index(data, fs=128, prev_value=None):
    """Calculate engagement index: (theta + alpha) / beta."""
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, alpha, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return (theta + alpha) / (beta + 1e-10)

def phase_amplitude_coupling(data, fs=128, prev_pac=None):
    """Calculate phase-amplitude coupling for theta-alpha, theta-beta, alpha-beta."""
    # Ensure correct shape for the filter (1 x samples x 1)
    data_reshaped = data[np.newaxis, :, np.newaxis]
    theta = butterworth_filter(data_reshaped, fs, 4.0, 8.0)[0, :, 0]
    alpha = butterworth_filter(data_reshaped, fs, 8.0, 13.0)[0, :, 0]
    beta = butterworth_filter(data_reshaped, fs, 13.0, 30.0)[0, :, 0]
    
    if np.any(np.isnan([theta, alpha, beta])) or np.std(theta) == 0 or np.std(alpha) == 0 or np.std(beta) == 0:
        return prev_pac if prev_pac is not None else np.zeros(3)
        
    theta_phase = np.angle(hilbert(theta))
    alpha_phase = np.angle(hilbert(alpha))
    beta_phase = np.angle(hilbert(beta))
    
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    
    pac_theta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase)))
    pac_theta_beta = np.abs(np.mean(beta_amp * np.exp(1j * theta_phase)))
    pac_alpha_beta = np.abs(np.mean(beta_amp * np.exp(1j * alpha_phase)))
    
    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def wavelet_features(data, prev_wavelet=None):
    """Extract wavelet features using Mexican Hat wavelet (CWT)."""
    scales = np.arange(1, 4)
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    if np.any(np.isnan(coeffs)):
        return prev_wavelet if prev_wavelet is not None else np.zeros(6)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.max(np.abs(c))])
    return np.array(features)

def theta_beta_ratio(data, fs=128, prev_value=None):
    """Calculate theta/beta ratio."""
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return theta / (beta + 1e-10)

def channel_coherence(data, fs=128, prev_coherence=None):
    """Calculate coherence between channel pairs in theta, alpha, and beta bands."""
    n_channels = data.shape[1]
    coherence_features = []
    expected_length = 3 * (n_channels * (n_channels - 1) // 2)
    nperseg = min(len(data), 512)
    
    for band, freq_range in [('theta', (4, 8)), ('alpha', (8, 13)), ('beta', (13, 30))]:
        for i in range(n_channels):
            for j in range(i + 1, n_channels):
                f, coh = signal.coherence(data[:, i], data[:, j], fs=fs, nperseg=nperseg)
                freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
                coh_value = np.mean(coh[freq_mask]) if freq_mask.any() else 0
                if np.isnan(coh_value):
                    return prev_coherence if prev_coherence is not None else np.zeros(expected_length)
                coherence_features.append(coh_value)
    return np.array(coherence_features)

def channel_correlation(data, prev_correlation=None):
    """Calculate Pearson correlation coefficient between channel pairs."""
    n_channels = data.shape[1]
    corr_features = []
    expected_length = n_channels * (n_channels - 1) // 2
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            corr = np.corrcoef(data[:, i], data[:, j])[0, 1]
            if np.isnan(corr):
                return prev_correlation if prev_correlation is not None else np.zeros(expected_length)
            corr_features.append(corr)
    return np.array(corr_features)

def microstate_analysis(data, fs=128, n_states=4, prev_microstates=None):
    """Perform EEG microstate analysis using K-means clustering."""
    expected_length = n_states + n_states + 2 # Durations + Coverages + GFP (mean, std)
    gfp = np.std(data, axis=1)
    if np.any(np.isnan(gfp)):
        return prev_microstates if prev_microstates is not None else np.zeros(expected_length)
        
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
        
    try:
        kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10)
        kmeans.fit(peak_data)
        templates = kmeans.cluster_centers_
        
        distances = np.zeros((data.shape[0], n_states))
        for i, template in enumerate(templates):
            template_norm = np.linalg.norm(template)
            data_norm = np.linalg.norm(data, axis=1)
            distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
            
        labels = np.argmax(distances, axis=1)
        durations = np.zeros(n_states)
        gfp_peaks = gfp[peaks] if len(peaks) > 0 else gfp
        gfp_mean = np.mean(gfp_peaks)
        gfp_std = np.std(gfp_peaks)
        
        for state in range(n_states):
            state_indices = np.where(labels == state)[0]
            if len(state_indices) > 0:
                diff = np.diff(state_indices)
                breaks = np.where(diff > 1)[0]
                segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
                durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
                
        coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
        microstate_features = np.hstack([durations, coverage, [gfp_mean, gfp_std]])
        return microstate_features
    except Exception:
        return prev_microstates if prev_microstates is not None else np.zeros(expected_length)

def extract_features_single(sample, idx, prev_features=None):
    """Extract all features for a single multi-channel EEG segment."""
    n_channels = sample.shape[1]
    
    # Standardize data per segment
    normalized = np.zeros_like(sample)
    for ch in range(n_channels):
        channel_data = sample[:, ch]
        std = np.std(channel_data)
        if std > 0:
            normalized[:, ch] = (channel_data - np.mean(channel_data)) / std
        else:
            normalized[:, ch] = channel_data
            
    channel_features = []
    features_per_channel = 14
    total_per_channel = n_channels * features_per_channel
    microstate_len = 10
    corr_len = (n_channels * (n_channels - 1) // 2)
    coh_len = 3 * corr_len
    total_prev_len = total_per_channel + microstate_len + corr_len + coh_len
    
    if prev_features is not None and len(prev_features) != total_prev_len:
        raise ValueError(f"Expected prev_features length {total_prev_len}, got {len(prev_features)}")
        
    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        offset = ch * features_per_channel
        prev_values = prev_features[offset:offset + features_per_channel] if prev_features is not None else None
        
        engagement = calculate_engagement_index(channel_data, prev_value=prev_values[0] if prev_values is not None else None)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data, prev_pac=prev_values[4:7] if prev_values is not None else None)
        wavelet = wavelet_features(channel_data, prev_wavelet=prev_values[7:13] if prev_values is not None else None)
        tbr = theta_beta_ratio(channel_data, prev_value=prev_values[13] if prev_values is not None else None)
        
        ch_features = np.concatenate([
            [engagement], [theta], [alpha], [beta], pac, wavelet, [tbr]
        ])
        channel_features.append(ch_features)
        
    per_channel_features = np.array(channel_features).flatten()
    
    # Extract global and connectivity features
    microstates = microstate_analysis(normalized, prev_microstates=prev_features[total_per_channel:total_per_channel + microstate_len] if prev_features is not None else None)
    correlations = channel_correlation(normalized, prev_correlation=prev_features[total_per_channel + microstate_len:total_per_channel + microstate_len + corr_len] if prev_features is not None else None)
    coherences = channel_coherence(normalized, prev_coherence=prev_features[total_per_channel + microstate_len + corr_len:] if prev_features is not None else None)
    
    sample_features = np.concatenate([per_channel_features, microstates, correlations, coherences])
    return sample_features

def extract_all_features(data, chunk_size=100):
    """Extract features for all EEG samples."""
    start_time = time.time()
    n_samples, n_samples_per_segment, n_channels = data.shape
    feature_matrix = []
    prev_features = None
    
    for i in tqdm(range(n_samples), desc="Extracting features"):
        sample_features = extract_features_single(data[i], i, prev_features)
        feature_matrix.append(sample_features)
        prev_features = sample_features
        
    feature_matrix = np.array(feature_matrix)
    print(f"Feature matrix shape: {feature_matrix.shape}, Time={time.time()-start_time:.2f}s")
    return feature_matrix



## 5. Feature Selection (MRS-SHAP)

Implements SMOTE-ENN, Variance Thresholding, Correlation Filtering, and the proposed Multi-Objective Ranking Score (MRS-SHAP).


In [ ]:
import time
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from dtaidistance import dtw
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from imblearn.combine import SMOTEENN
import psutil

def select_top_features(scores, percentage=0.6):
    """Retain the top percentage of features based on scores."""
    k = max(1, int(len(scores) * percentage))
    return np.argsort(scores)[::-1][:k]

def remove_highly_correlated_features(X, feature_names, threshold=0.85):
    """Remove highly correlated features (Pearson correlation > threshold)."""
    corr_matrix = np.abs(np.corrcoef(X.T))
    to_remove = set()
    for i in range(len(corr_matrix)):
        for j in range(i + 1, len(corr_matrix)):
            if corr_matrix[i, j] > threshold:
                to_remove.add(j)
    keep_indices = [i for i in range(X.shape[1]) if i not in to_remove]
    return X[:, keep_indices], np.array(feature_names)[keep_indices].tolist()

def shap_selection(X, y):
    """Estimate feature importance using mean absolute SHAP values from XGBoost."""
    xgb_model = xgb.XGBClassifier(random_state=42, n_estimators=100, eval_metric='logloss', reg_lambda=0.1)
    xgb_model.fit(X, y)
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X)
    if isinstance(shap_values, list):
        # In binary classification, SHAP values are a list of length 2
        shap_values = shap_values[1]
    shap_importance = np.abs(shap_values).mean(axis=0)
    return shap_importance

def dtw_selection(X, subject_ids, min_window_size=50):
    """Calculate temporal stability scores using Dynamic Time Warping (DTW)."""
    dtw_scores = np.zeros(X.shape[1])
    for feature_idx in range(X.shape[1]):
        feature_series = X[:, feature_idx]
        if np.any(np.isnan(feature_series)) or np.std(feature_series) == 0:
            dtw_scores[feature_idx] = 0
            continue
        
        # Segment the feature series into sub-windows
        sub_windows = [feature_series[i:i+min_window_size] for i in range(0, len(feature_series)-min_window_size, min_window_size)]
        if len(sub_windows) < 2:
            dtw_scores[feature_idx] = 0
            continue
            
        try:
            dtw_sum = 0
            count = 0
            # Pairwise DTW alignment
            for i in range(len(sub_windows)):
                for j in range(i+1, len(sub_windows)):
                    dist = dtw.distance(sub_windows[i], sub_windows[j])
                    if np.isnan(dist):
                        continue
                    dtw_sum += dist
                    count += 1
            dtw_scores[feature_idx] = dtw_sum / count if count > 0 else 0
        except Exception:
            dtw_scores[feature_idx] = 0
            
    # Stability: invert the DTW alignment cost (smaller cost means higher stability)
    # Normalize stable features to have high values
    dtw_inverted = np.max(dtw_scores) - dtw_scores
    return dtw_inverted

def mrs_selection(shap_scores, dtw_scores, w_shap=0.6, w_dtw=0.4, BW=0.0):
    """Calculate the Multi-Objective Ranking Score (MRS) for feature selection."""
    sn = (shap_scores - shap_scores.min()) / (shap_scores.max() - shap_scores.min() + 1e-10)
    dn = (dtw_scores - dtw_scores.min()) / (dtw_scores.max() - dtw_scores.min() + 1e-10)
    
    # Equation 21: suboptimal model performance amplifies SHAP weighting
    mo_score = w_shap * sn * (1.0 + BW) + w_dtw * dn
    return mo_score

def select_features_mrs_shap_full(X, y, feature_names, subject_ids, n_features_to_select=0.6, min_window_size=50, BW=0.0):
    """Wrapper function to perform the full MRS-SHAP feature selection pipeline."""
    start_time = time.time()
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1024**2

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    # SMOTE-ENN boundary cleaning
    smote_enn = SMOTEENN(random_state=42, sampling_strategy='auto')
    X_balanced, y_balanced = smote_enn.fit_resample(X_scaled, y)

    # Variance Thresholding
    selector = VarianceThreshold(threshold=0.01)
    X_reduced = selector.fit_transform(X_balanced)
    mask = selector.get_support()
    feature_names_reduced = np.array(feature_names)[mask].tolist()

    # Correlation Filtering
    X_reduced, feature_names_reduced = remove_highly_correlated_features(X_reduced, feature_names_reduced, threshold=0.85)

    # Compute SHAP and DTW stability
    shap_scores = shap_selection(X_reduced, y_balanced)
    
    # DTW uses raw/un-oversampled data for accurate subject-level temporal stability
    X_reduced_raw = X_scaled[:, [feature_names.index(f) for f in feature_names_reduced]]
    dtw_scores = dtw_selection(X_reduced_raw, subject_ids, min_window_size)
    
    # Compute MRS
    mrs_scores = mrs_selection(shap_scores, dtw_scores, w_shap=0.6, w_dtw=0.4, BW=BW)
    selected_indices = select_top_features(mrs_scores, percentage=n_features_to_select)
    
    selected_features = [feature_names_reduced[idx] for idx in selected_indices]
    selected_indices_orig = [feature_names.index(f) for f in selected_features]
    
    mem_after = process.memory_info().rss / 1024**2
    print(f"MRS-SHAP complete: Features selected={len(selected_features)}, Time={time.time()-start_time:.2f}s, Memory={mem_after-mem_before:.2f}MB")
    
    return selected_features, X[:, selected_indices_orig]



## 6. Machine Learning Models & Ensemble

Defines the classifiers and the soft Voting Ensemble matching the manuscript's configuration.


In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from catboost import CatBoostClassifier

def get_base_models():
    """Return base machine learning models with hyperparameters matching the manuscript."""
    # Support Vector Machine (SVM)
    svm = SVC(
        C=10.0,
        kernel='rbf',
        gamma='scale',
        probability=True,
        class_weight='balanced',
        random_state=42
    )
    
    # Logistic Regression (LR)
    lr = LogisticRegression(
        C=1.0,
        penalty='l2',
        solver='liblinear',
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
    
    # Gradient Boosting (GB)
    gb = GradientBoostingClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        random_state=42
    )
    
    # CatBoost Classifier
    cat = CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        l2_leaf_reg=5,
        auto_class_weights='Balanced',
        verbose=0,
        random_state=42
    )
    
    return {
        'svm': svm,
        'lr': lr,
        'gb': gb,
        'cat': cat
    }

def build_voting_ensemble(models=None):
    """Build the soft Voting Ensemble model using SVM, LR, GB, and CatBoost."""
    if models is None:
        models = get_base_models()
        
    ensemble = VotingClassifier(
        estimators=[
            ('cat', models['cat']),
            ('svm', models['svm']),
            ('gb', models['gb']),
            ('lr', models['lr'])
        ],
        voting='soft',
        # CatBoost (0.3), SVM (0.3), GB (0.25), LR (0.15)
        weights=[0.3, 0.3, 0.25, 0.15]
    )
    return ensemble



## 7. Pipeline Execution

Orchestrates dataset loading (or generation of synthetic data in simulation mode), subject-level cross-validation, and held-out test evaluation.


In [ ]:
import os
import time
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import loadmat
import shap

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from imblearn.combine import SMOTEENN

# Import config parameters
import config

# Import pipeline steps
from preprocessing import (
    notch_filter, clip_spikes, butterworth_filter,
    find_optimal_weights, apply_ica, compare_ica_methods
)
from feature_extraction import extract_all_features
from feature_selection import (
    select_top_features, remove_highly_correlated_features,
    shap_selection, dtw_selection, mrs_selection
)
from modeling import get_base_models, build_voting_ensemble

# ==========================================
# DATA LOADING UTILITIES
# ==========================================

def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    """Load a single MAT file and segment into overlapping windows."""
    try:
        mat_data = loadmat(file_path)
        # Exclude metadata variables
        var_names = [k for k in mat_data.keys() if not k.startswith('__')]
        if not var_names:
            return None
        data = mat_data[var_names[0]]

        if data.ndim == 2:
            data = data[np.newaxis, :, :] # Add segment dimension if 2D

        segments = []
        for seg in data:
            # Segment along the time dimension (axis 0 of the segment)
            for start in range(0, seg.shape[0] - window_size, stride):
                window = seg[start:start + window_size, :]
                if window.shape[1] == expected_channels:
                    segments.append(window)

        return np.array(segments) if segments else None
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def load_from_folder_with_ids(folder_path, label, window_size=512, stride=256, expected_channels=19):
    """Load and segment all MAT files in a folder and associate patient IDs."""
    if not os.path.exists(folder_path):
        print(f"Directory not found: {folder_path}")
        return None, None, None

    files = sorted([f for f in os.listdir(folder_path) if f.endswith('.mat')])
    if not files:
        return None, None, None

    all_segments = []
    patient_ids = []
    labels = []

    for f in files:
        file_path = os.path.join(folder_path, f)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None and len(segments) > 0:
            all_segments.append(segments)
            pid = f.split(".")[0]
            patient_ids.extend([pid] * len(segments))
            labels.extend([label] * len(segments))

    if not all_segments:
        return None, None, None

    return np.concatenate(all_segments), patient_ids, labels

def load_dataset(window_size=512, stride=256, expected_channels=19):
    """Load ADHD and Control data from all configured paths."""
    print("Loading resting-state EEG dataset...")
    
    # Load ADHD parts
    X_a1, pids_a1, y_a1 = load_from_folder_with_ids(config.ADHD_PART1, 1, window_size, stride, expected_channels)
    X_a2, pids_a2, y_a2 = load_from_folder_with_ids(config.ADHD_PART2, 1, window_size, stride, expected_channels)
    
    # Load Control parts
    X_c1, pids_c1, y_c1 = load_from_folder_with_ids(config.CONTROL_PART1, 0, window_size, stride, expected_channels)
    X_c2, pids_c2, y_c2 = load_from_folder_with_ids(config.CONTROL_PART2, 0, window_size, stride, expected_channels)

    # Combine parts
    X_list, pids_list, y_list = [], [], []
    for dataset in [(X_a1, pids_a1, y_a1), (X_a2, pids_a2, y_a2), (X_c1, pids_c1, y_c1), (X_c2, pids_c2, y_c2)]:
        if dataset[0] is not None:
            X_list.append(dataset[0])
            pids_list.extend(dataset[1])
            y_list.extend(dataset[2])

    if not X_list:
        raise FileNotFoundError("No EEG data found. Check your config.py dataset directories or run with --simulation.")

    X = np.concatenate(X_list)
    y = np.array(y_list)
    patient_ids = np.array(pids_list)

    print(f"Total segments loaded : {X.shape[0]}")
    print(f"Unique subjects        : {len(np.unique(patient_ids))}")
    print(f"ADHD segments (Class 1): {np.sum(y == 1)}")
    print(f"Control segments (Class 0): {np.sum(y == 0)}")
    return X, y, patient_ids

# ==========================================
# SIMULATION DATA GENERATION (DRY RUN)
# ==========================================

def generate_simulation_data(n_subjects=30, segments_per_subject=15):
    """Generate synthetic EEG data for dry run testing of the pipeline."""
    print("Generating simulation EEG data...")
    n_samples = config.WINDOW_SIZE
    n_channels = config.EXPECTED_CHANNELS
    
    X_list, pids_list, y_list = [], [], []
    
    for s_idx in range(n_subjects):
        subject_id = f"Sub{s_idx:03d}"
        # Split subjects equally between ADHD (1) and Control (0)
        label = 1 if s_idx < n_subjects // 2 else 0
        
        # Generate random segments with differing frequency attributes
        for _ in range(segments_per_subject):
            if label == 1:
                # ADHD has slightly elevated slow waves (4-8Hz theta)
                t = np.linspace(0, n_samples/config.FS, n_samples)
                sig = np.sin(2 * np.pi * 6 * t)[:, np.newaxis] * 1.5 + np.random.randn(n_samples, n_channels)
            else:
                # Controls have normal alpha/beta
                t = np.linspace(0, n_samples/config.FS, n_samples)
                sig = np.sin(2 * np.pi * 10 * t)[:, np.newaxis] * 1.2 + np.random.randn(n_samples, n_channels)
                
            X_list.append(sig)
            pids_list.append(subject_id)
            y_list.append(label)
            
    X = np.array(X_list)
    y = np.array(y_list)
    patient_ids = np.array(pids_list)
    
    print(f"Simulated segments : {X.shape[0]}")
    print(f"Simulated subjects  : {len(np.unique(patient_ids))}")
    return X, y, patient_ids

# ==========================================
# MAIN EXECUTION PIPELINE
# ==========================================

def run_experiment(X, y, patient_ids, run_full_features=False):
    """Run preprocessing, feature extraction, selection, CV, and test set evaluation."""
    # Step 1: Preprocessing
    print("\n--- Preprocessing ---")
    print("Applying notch and Butterworth bandpass filters...")
    X_pre = notch_filter(X, config.FS, freqs=[50, 60], Q=30)
    X_pre = clip_spikes(X_pre, threshold=500)
    X_pre = butterworth_filter(X_pre, config.FS, lowcut=1.0, highcut=45.0, order=4)

    print("Searching for optimal Weighted ICA weights...")
    # Find optimal weights on a subset of data for execution speed
    subset_idx = np.random.choice(len(X_pre), min(10, len(X_pre)), replace=False)
    best_weights, _, _ = find_optimal_weights(X_pre[subset_idx], X[subset_idx], fs=config.FS, n_components=config.EXPECTED_CHANNELS)
    
    print("Applying weighted ICA artifact removal...")
    X_cleaned, _ = apply_ica(X_pre, fs=config.FS, method='weighted', weights=best_weights)

    # Step 2: Feature Extraction
    print("\n--- Feature Extraction ---")
    X_features = extract_all_features(X_cleaned)
    print(f"Extracted feature matrix shape: {X_features.shape}")

    # Build feature name list matching the 998 feature set
    n_channels = config.EXPECTED_CHANNELS
    feature_names = []
    for ch in range(n_channels):
        feature_names.extend([
            f'engagement_ch{ch}', f'theta_ch{ch}', f'alpha_ch{ch}', f'beta_ch{ch}',
            f'pac_theta_alpha_ch{ch}', f'pac_theta_beta_ch{ch}', f'pac_alpha_beta_ch{ch}',
            f'wavelet_mean1_ch{ch}', f'wavelet_max1_ch{ch}',
            f'wavelet_mean2_ch{ch}', f'wavelet_max2_ch{ch}',
            f'wavelet_mean3_ch{ch}', f'wavelet_max3_ch{ch}',
            f'theta_beta_ratio_ch{ch}'
        ])
    microstate_names = [f'microstate_duration_{i}' for i in range(4)] + [f'microstate_coverage_{i}' for i in range(4)] + ['gfp_mean', 'gfp_std']
    corr_names = [f'corr_ch{i}_ch{j}' for i in range(n_channels) for j in range(i+1, n_channels)]
    coh_names = [f'coh_{band}_ch{i}_ch{j}' for band in ['theta', 'alpha', 'beta'] for i in range(n_channels) for j in range(i+1, n_channels)]
    feature_names.extend(microstate_names + corr_names + coh_names)

    # Step 3: Train/Test Split (Subject-Level)
    print("\n--- Train/Test Split (Subject-Level) ---")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X_features, y, groups=patient_ids))

    X_train_all, X_test_held = X_features[train_idx], X_features[test_idx]
    y_train_all, y_test_held = y[train_idx], y[test_idx]
    sids_train = patient_ids[train_idx]
    sids_test = patient_ids[test_idx]

    print(f"Training subjects      : {len(np.unique(sids_train))} ({len(X_train_all)} segments)")
    print(f"Held-out test subjects : {len(np.unique(sids_test))} ({len(X_test_held)} segments)")

    # Step 4: Subject-Level 5-Fold CV (Outer Loop)
    print("\n--- Running Subject-Level 5-Fold CV (Leakage-Free) ---")
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    cv_results = []
    
    # Store selected features for test set evaluation
    fold_selected_features = []

    for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X_train_all, y_train_all, groups=sids_train), 1):
        X_tr_raw, X_te = X_train_all[tr_idx], X_train_all[te_idx]
        y_tr_raw, y_te = y_train_all[tr_idx], y_train_all[te_idx]
        sids_tr = sids_train[tr_idx]

        # Inner-loop 5-fold CV on training fold to compute metrics for Equation 21 (BW)
        print(f"Fold {fold}: Computing inner-loop metrics for Balanced Weight (BW)...")
        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        inner_metrics = []
        
        for val_tr_idx, val_te_idx in inner_cv.split(X_tr_raw, y_tr_raw):
            X_val_tr_raw, X_val_te = X_tr_raw[val_tr_idx], X_tr_raw[val_te_idx]
            y_val_tr_raw, y_val_te = y_tr_raw[val_tr_idx], y_tr_raw[val_te_idx]
            
            smote_enn = SMOTEENN(random_state=42)
            X_val_tr, y_val_tr = smote_enn.fit_resample(X_val_tr_raw, y_val_tr_raw)
            
            # Simple modeling in inner loop for speed
            models = get_base_models()
            inner_clf = models['cat']
            inner_clf.fit(X_val_tr[:, :50], y_val_tr) # Evaluate on a subset of features for speed
            preds = inner_clf.predict(X_val_te[:, :50])
            
            cm_val = confusion_matrix(y_val_te, preds)
            tn, fp, fn, tp = cm_val.ravel()
            sens = tp / (tp + fn + 1e-8)
            spec = tn / (tn + fp + 1e-8)
            acc = accuracy_score(y_val_te, preds)
            f1 = f1_score(y_val_te, preds)
            inner_metrics.append([sens, spec, f1, acc])
            
        mean_metrics = np.mean(inner_metrics, axis=0)
        
        # Calculate BW using Equations 18-20 (WITHOUT division by sum)
        metrics_norm = (mean_metrics - mean_metrics.min()) / (mean_metrics.max() - mean_metrics.min() + 1e-10)
        metric_weights = 1.0 - metrics_norm
        BW = np.mean(metric_weights)
        print(f"Fold {fold} Balanced Weight (BW): {BW:.4f}")

        # Standardize training data per fold
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr_raw)
        X_tr_scaled = np.nan_to_num(X_tr_scaled, nan=0.0, posinf=0.0, neginf=0.0)

        # Apply SMOTE-ENN boundary refinement
        smote_enn = SMOTEENN(random_state=42)
        X_tr_bal, y_tr_bal = smote_enn.fit_resample(X_tr_scaled, y_tr_raw)

        # Variance thresholding
        var_sel = VarianceThreshold(threshold=0.01)
        X_tr_red = var_sel.fit_transform(X_tr_bal)
        support_mask = var_sel.get_support()
        feat_names_red = np.array(feature_names)[support_mask].tolist()

        # Correlation filtering
        X_tr_red, feat_names_red = remove_highly_correlated_features(X_tr_red, feat_names_red, threshold=0.85)

        # Feature selection using MRS-SHAP (incorporating BW)
        shap_scores = shap_selection(X_tr_red, y_tr_bal)
        
        X_tr_raw_red = X_tr_scaled[:, [feature_names.index(f) for f in feat_names_red]]
        dtw_scores = dtw_selection(X_tr_raw_red, sids_tr)
        
        mrs_scores = mrs_selection(shap_scores, dtw_scores, w_shap=0.6, w_dtw=0.4, BW=BW)
        selected_idx = select_top_features(mrs_scores, percentage=0.6)
        
        selected_feats = [feat_names_red[idx] for idx in selected_idx]
        fold_selected_features.append(selected_feats)
        
        # Train and evaluate CatBoost classifier on selected features
        selected_indices_orig = [feature_names.index(f) for f in selected_feats]
        
        # Scale test fold
        X_te_scaled = scaler.transform(X_te)
        X_te_scaled = np.nan_to_num(X_te_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        
        model = get_base_models()['cat']
        model.fit(X_tr_bal[:, [feat_names_red.index(f) for f in selected_feats]], y_tr_bal)
        
        y_pred = model.predict(X_te_scaled[:, selected_indices_orig])
        y_prob = model.predict_proba(X_te_scaled[:, selected_indices_orig])[:, 1]

        cm = confusion_matrix(y_te, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fold_sens = tp / (tp + fn + 1e-8)
        fold_spec = tn / (tn + fp + 1e-8)
        fold_acc = accuracy_score(y_te, y_pred)
        fold_f1 = f1_score(y_te, y_pred)
        fold_auc = roc_auc_score(y_te, y_prob)

        print(f"Fold {fold} - Accuracy: {fold_acc:.4f}, F1: {fold_f1:.4f}, AUC: {fold_auc:.4f}")
        cv_results.append([fold, fold_sens, fold_spec, fold_f1, fold_acc, fold_auc])

    df_cv = pd.DataFrame(cv_results, columns=["Fold", "Sensitivity", "Specificity", "F1", "Accuracy", "AUC"])
    print("\n--- Average Subject-Level CV Performance ---")
    print(df_cv.mean().round(4))

    # Step 5: Final Evaluation on Held-out Test Set using Voting Ensemble
    print("\n--- Final Evaluation on Held-out Test Set ---")
    # Fit MRS-SHAP on full training set
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_all)
    X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    
    # SMOTE-ENN
    smote_enn = SMOTEENN(random_state=42)
    X_train_bal, y_train_bal = smote_enn.fit_resample(X_train_scaled, y_train_all)
    
    # Variance and correlation
    var_sel = VarianceThreshold(threshold=0.01)
    X_train_red = var_sel.fit_transform(X_train_bal)
    support_mask = var_sel.get_support()
    feat_names_red = np.array(feature_names)[support_mask].tolist()
    X_train_red, feat_names_red = remove_highly_correlated_features(X_train_red, feat_names_red, threshold=0.85)

    # Compute final select indices
    shap_scores = shap_selection(X_train_red, y_train_bal)
    X_train_raw_red = X_train_scaled[:, [feature_names.index(f) for f in feat_names_red]]
    dtw_scores = dtw_selection(X_train_raw_red, sids_train)
    mrs_scores = mrs_selection(shap_scores, dtw_scores, w_shap=0.6, w_dtw=0.4, BW=0.25)
    selected_idx = select_top_features(mrs_scores, percentage=0.6)
    selected_feats = [feat_names_red[idx] for idx in selected_idx]
    
    # Scale test set
    X_test_scaled = scaler.transform(X_test_held)
    X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    selected_indices_orig = [feature_names.index(f) for f in selected_feats]
    X_tr_ensemble = X_train_bal[:, [feat_names_red.index(f) for f in selected_feats]]
    X_te_ensemble = X_test_scaled[:, selected_indices_orig]

    # Train ensemble model
    ensemble = build_voting_ensemble()
    ensemble.fit(X_tr_ensemble, y_train_bal)
    
    y_pred_held = ensemble.predict(X_te_ensemble)
    y_prob_held = ensemble.predict_proba(X_te_ensemble)[:, 1]

    cm_held = confusion_matrix(y_test_held, y_pred_held)
    tn, fp, fn, tp = cm_held.ravel()
    sens_held = tp / (tp + fn + 1e-8)
    spec_held = tn / (tn + fp + 1e-8)
    acc_held = accuracy_score(y_test_held, y_pred_held)
    f1_held = f1_score(y_test_held, y_pred_held)
    auc_held = roc_auc_score(y_test_held, y_prob_held)

    print("\nHeld-out Test Set Performance:")
    print(f"Confusion Matrix:\n{cm_held}")
    print(f"Sensitivity: {sens_held:.4f}")
    print(f"Specificity: {spec_held:.4f}")
    print(f"F1-Score: {f1_held:.4f}")
    print(f"Accuracy: {acc_held:.4f}")
    print(f"AUC: {auc_held:.4f}")

    # Generate Figures
    print("\nSaving figures...")
    # 1. Confusion Matrix
    plt.figure(figsize=(6, 5), dpi=300)
    sns.heatmap(cm_held, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix (Test Set)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig('confusion_matrix_test.png')
    plt.close()

    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_test_held, y_prob_held)
    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(fpr, tpr, color='orange', label=f'ROC Curve (AUC = {auc_held:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title('ROC Curve (Test Set)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('roc_curve_test.png')
    plt.close()

    # Save outputs
    print("\nSaving selected features list to 'selected_features.txt'...")
    with open('selected_features.txt', 'w') as f:
        f.write('\n'.join(selected_feats))

    # Save final test metrics
    final_metrics_df = pd.DataFrame({
        'Metric': ['Sensitivity', 'Specificity', 'F1-Score', 'Accuracy', 'AUC'],
        'Value': [sens_held, spec_held, f1_held, acc_held, auc_held]
    })
    final_metrics_df.to_csv('final_test_metrics.csv', index=False)
    print("Final test metrics saved to 'final_test_metrics.csv'")
    print("✔ Pipeline execution completed successfully!")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Run the MRS-SHAP ADHD EEG pipeline.")
    parser.add_argument('--simulation', action='store_true', help='Run pipeline with simulated data for test/dry-run.')
    args = parser.parse_args()

    if args.simulation:
        X, y, patient_ids = generate_simulation_data(n_subjects=10, segments_per_subject=5)
    else:
        X, y, patient_ids = load_dataset(config.WINDOW_SIZE, config.STRIDE, config.EXPECTED_CHANNELS)
        
    run_experiment(X, y, patient_ids)

